### Imports

In [72]:
import pickle
from copy import copy

import re
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.normalizers import Lowercase

from transformers import PreTrainedTokenizerFast

### Dataset loading

In [2]:
with open("data\\pos_data.txt", 'r', encoding = "utf-8") as f:
    data = list(filter(None, [line.rstrip("\n") for line in f]))

In [3]:
def merge_sentences(data: List[str]) -> List[List[Tuple[str, str]]]:
    """Merges sentences from the given list of lines"""
    sentences = []
    current_sentence = []
    for row in data:
        word_tag = re.search("\t.*?\t", row)[0].strip("\t")
        pos = re.search("[A-z]+", row)[0].strip("|") if word_tag not in ["<end>", "<beg>"] else re.search("\t.*?\t", row)[0].strip("\t")
        current_sentence.append((word_tag, pos))
        if current_sentence[0][0] == "<beg>" and current_sentence[-1][0] == "<end>":
            sentences.append(copy(current_sentence))
            current_sentence.clear()
        else:
            continue
    return sentences


In [4]:
processed_sentences = merge_sentences(data)
processed_sentences

[[('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('колодец', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'), ('рой', 'VERB'), ('погреб', 'NOUN'), ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('укрытие', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('я', 'PRON'),
  ('мою', 'VERB'),
  ('окно', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('сотрудники', 'NOUN'),
  ('милиции', 'NOUN'),
  ('вечером', 'NOUN'),
  ('31', 'NUM'),
  ('декабря', 'NOUN'),
  ('уничтожили', 'VERB'),
  ('в', 'ADP'),
  ('хасавюрте', 'NOUN'),
  ('четверых', 'NUM'),
  ('боевиков', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('вместе', 'ADV'),
  ('с', 'ADP'),
  ('ним', 'PRON'),
  ('берлускони', 'NOUN'),
  ('выпустил', 'VERB'),
  ('уже', 'ADV'),
  ('четыре', 'NUM'),
  ('пластинки', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('психически', 'ADV'),
  ('неуравновешенный', 'ADJ'),
  ('житель', 'NOUN'),
  ('милана', 'NOUN'),
 

In [5]:
# Saved processed dataset
# with open('artifacts\\pos_processed_dataset.pkl', 'wb') as file:
#     pickle.dump(processed_sentences, file)

In [6]:
# Load processed dataset
with open('artifacts\\pos_processed_dataset.pkl', 'rb') as file:
    processed_dataset = pickle.load(file)

In [25]:
processed_dataset

[[('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('колодец', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'), ('рой', 'VERB'), ('погреб', 'NOUN'), ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('укрытие', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('я', 'PRON'),
  ('мою', 'VERB'),
  ('окно', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('сотрудники', 'NOUN'),
  ('милиции', 'NOUN'),
  ('вечером', 'NOUN'),
  ('31', 'NUM'),
  ('декабря', 'NOUN'),
  ('уничтожили', 'VERB'),
  ('в', 'ADP'),
  ('хасавюрте', 'NOUN'),
  ('четверых', 'NUM'),
  ('боевиков', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('вместе', 'ADV'),
  ('с', 'ADP'),
  ('ним', 'PRON'),
  ('берлускони', 'NOUN'),
  ('выпустил', 'VERB'),
  ('уже', 'ADV'),
  ('четыре', 'NUM'),
  ('пластинки', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('психически', 'ADV'),
  ('неуравновешенный', 'ADJ'),
  ('житель', 'NOUN'),
  ('милана', 'NOUN'),
 

In [50]:
words = [item[0] for sentence_list in processed_dataset for item in sentence_list]
words

['<beg>',
 'рой',
 'колодец',
 '<end>',
 '<beg>',
 'рой',
 'погреб',
 '<end>',
 '<beg>',
 'рой',
 'укрытие',
 '<end>',
 '<beg>',
 'я',
 'мою',
 'окно',
 '<end>',
 '<beg>',
 'сотрудники',
 'милиции',
 'вечером',
 '31',
 'декабря',
 'уничтожили',
 'в',
 'хасавюрте',
 'четверых',
 'боевиков',
 '.',
 '<end>',
 '<beg>',
 'вместе',
 'с',
 'ним',
 'берлускони',
 'выпустил',
 'уже',
 'четыре',
 'пластинки',
 '.',
 '<end>',
 '<beg>',
 'психически',
 'неуравновешенный',
 'житель',
 'милана',
 'швырнул',
 'в',
 'политика',
 'тяжелую',
 'сувенирную',
 'статуэтку',
 '.',
 '<end>',
 '<beg>',
 'в',
 'марте',
 '2009',
 'года',
 'литва',
 'подписала',
 'десятилетний',
 'контракт',
 'на',
 'поставки',
 'электроэнергии',
 'из',
 'россии',
 '.',
 '<end>',
 '<beg>',
 'атмосферу',
 'на',
 'улицах',
 'сотрудники',
 'штаба',
 'охарактеризовали',
 'как',
 'праздничную',
 '.',
 '<end>',
 '<beg>',
 'бельгиец',
 'будет',
 'занимать',
 'пост',
 'президента',
 'ес',
 'два',
 'с',
 'половиной',
 'года',
 '.',
 '<end

In [52]:
words.extend(["<pad>", "<unk>"])

In [34]:
main_tags = set(item[1] for sentence_list in processed_dataset for item in sentence_list)
main_tags

{'hennessy',
 'form',
 'ego',
 'supreme',
 'vladimir',
 'sire',
 'freescale',
 'seine',
 'urbach',
 'printemps',
 'sai',
 'pharma',
 'best',
 'htm',
 'punica',
 'jaguar',
 'mathcad',
 'liveness',
 'esposito',
 'tone',
 'metrix',
 'fortunes',
 'akira',
 'osraige',
 'feelgood',
 'billboard',
 'synchronous',
 'relic',
 'concertgebouw',
 'cosmopolitan',
 'nine',
 'ads',
 'logan',
 'buy',
 'offence',
 'columbia',
 'mora',
 'remittances',
 'gcp',
 'gobject',
 'central',
 'massive',
 'baso',
 'lettereng',
 'gcpconf',
 'cnn',
 'porco',
 'starbucks',
 'sportbladet',
 'test',
 'take',
 'seeo',
 'spartan',
 'communications',
 'kurz',
 'xn',
 'check',
 'ssd',
 'giro',
 'vergegenstandigung',
 'wyborcza',
 'bisque',
 'nyse',
 'capitals',
 'gtk',
 'californication',
 'tn',
 'target',
 'depater',
 'bacl',
 'grata',
 'sponte',
 'vii',
 'dirty',
 'telenor',
 'ignis',
 'edmond',
 'at',
 'lure',
 'grisea',
 'non',
 'silvano',
 'dffpc',
 'publishing',
 'bay',
 'rammstein',
 'inobedientia',
 'blonde',
 'ser

In [38]:
print(len(main_tags))

5206


In [41]:
# Create dict for class labels
key = 0
for label in main_tags:
    pass
label_dict = dict()
for tag in main_tags:
    label_dict[tag] = key
    key += 1

In [49]:
label_dict

{'hennessy': 0,
 'form': 1,
 'ego': 2,
 'supreme': 3,
 'vladimir': 4,
 'sire': 5,
 'freescale': 6,
 'seine': 7,
 'urbach': 8,
 'printemps': 9,
 'sai': 10,
 'pharma': 11,
 'best': 12,
 'htm': 13,
 'punica': 14,
 'jaguar': 15,
 'mathcad': 16,
 'liveness': 17,
 'esposito': 18,
 'tone': 19,
 'metrix': 20,
 'fortunes': 21,
 'akira': 22,
 'osraige': 23,
 'feelgood': 24,
 'billboard': 25,
 'synchronous': 26,
 'relic': 27,
 'concertgebouw': 28,
 'cosmopolitan': 29,
 'nine': 30,
 'ads': 31,
 'logan': 32,
 'buy': 33,
 'offence': 34,
 'columbia': 35,
 'mora': 36,
 'remittances': 37,
 'gcp': 38,
 'gobject': 39,
 'central': 40,
 'massive': 41,
 'baso': 42,
 'lettereng': 43,
 'gcpconf': 44,
 'cnn': 45,
 'porco': 46,
 'starbucks': 47,
 'sportbladet': 48,
 'test': 49,
 'take': 50,
 'seeo': 51,
 'spartan': 52,
 'communications': 53,
 'kurz': 54,
 'xn': 55,
 'check': 56,
 'ssd': 57,
 'giro': 58,
 'vergegenstandigung': 59,
 'wyborcza': 60,
 'bisque': 61,
 'nyse': 62,
 'capitals': 63,
 'gtk': 64,
 'califo

### Text tokenization

In [53]:
tokenizer = Tokenizer(model = BPE(unk_token = "<unk>"))
tokenizer.normalizer = Lowercase()

trainer = BpeTrainer(vocab_size = 30_000, special_tokens = ["<pad>", "<beg>", "<end>", "<unk>"])

In [54]:
tokenizer.train_from_iterator(iterator=words, trainer = trainer)

In [61]:
base_vocabulary = tokenizer.get_vocab()
base_vocabulary["<pad>"]

0

In [100]:
# calculate maximum length of sentences
max_length = 0
for sentence in processed_dataset:
    max_length = max(max_length, len(sentence))
max_length

472

In [94]:
main_tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer, pad_token = "<pad>")

In [ ]:
encoded_sentence = main_tokenizer("Рой", padding="max_length", max_length=max_length)["input_ids"]
encoded_sentence

[1017,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 

In [98]:
main_tokenizer.decode(encoded_sentence, skip_special_tokens = True)

'рой'

In [99]:
# calculate maximum length of sentences
main_tokenizer

TokenizersBackend(name_or_path='', vocab_size=30000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<beg>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<end>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

### Dataset preparation

In [158]:
class POSDataset:
    def __init__(self, dataset:List[List[Tuple[str, str]]],
                 tokenizer: PreTrainedTokenizerFast,
                 max_length: int = 472,
                 pad_id: int = 0):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pad_id = pad_id

    def __getitem__(self, idx):
        single_sentence = self.dataset[idx]
        tokenized_sentence = []
        token_classes = []
        for tup in single_sentence:
            tokenized_sentence.extend(self.tokenizer(tup[0])["input_ids"])
        for tup in single_sentence:
            token_classes.extend([label_dict[tup[1]]])
        while len(tokenized_sentence) < self.max_length:
            tokenized_sentence.append(self.pad_id)
        while len(token_classes) < self.max_length:
            token_classes.append(label_dict["x"])
        return torch.tensor(tokenized_sentence), torch.tensor(token_classes), len(token_classes)

    def __len__(self):
        return len(self.dataset)

In [172]:
pos_dataset = POSDataset(processed_dataset, main_tokenizer)
pos_dataset

In [175]:
# Those lables are going to be ignored
label_dict["x"]

4136

In [176]:
pos_dataset[23][0].shape

torch.Size([472])

### Model definition

In [165]:
len(main_tokenizer.get_vocab())

30000

In [229]:
class POSModel(nn.Module):
    def __init__(self, input_size:int, hidden_size:int,
                 num_embeddings:int = len(main_tokenizer.get_vocab()), embeddings_dim: int = 300,
                 bidirectional: bool = False, vocab_size: int = len(main_tokenizer.get_vocab())):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, input_size)
        self.rnn = nn.RNN(input_size, hidden_size)
        self.linear = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x):
        y = self.embedding(x)
        y = y.transpose(0, 1)
        out, h = self.rnn(y) # S x B x H
        y = self.linear(out)
        y = y.flatten(0, 1)
        return y

### Training

In [230]:
train_dataloader = DataLoader(dataset = pos_dataset, shuffle = True, batch_size = 32)
train_dataloader

In [231]:
training_sample = next(iter(train_dataloader))
print(f"Training sample shape: {training_sample[0].shape}")
training_sample

Training sample shape: torch.Size([32, 472])


[tensor([[    1,   149,   787,  ...,     0,     0,     0],
         [    1, 10722,  3243,  ...,     0,     0,     0],
         [    1,   730,  2118,  ...,     0,     0,     0],
         ...,
         [    1,   500,  1228,  ...,     0,     0,     0],
         [    1,  7118,  1120,  ...,     0,     0,     0],
         [    1,  9893,  7538,  ...,     0,     0,     0]]),
 tensor([[1974, 3528, 2073,  ..., 4136, 4136, 4136],
         [1974, 1155, 2073,  ..., 4136, 4136, 4136],
         [1974, 2886, 2393,  ..., 4136, 4136, 4136],
         ...,
         [1974, 3528, 3528,  ..., 4136, 4136, 4136],
         [1974, 1155, 2073,  ..., 4136, 4136, 4136],
         [1974, 1155, 1155,  ..., 4136, 4136, 4136]]),
 tensor([472, 472, 472, 472, 472, 472, 472, 472, 472, 472, 472, 472, 472, 472,
         472, 472, 472, 472, 472, 472, 472, 472, 472, 472, 472, 472, 472, 472,
         472, 472, 472, 472])]

In [232]:
model = POSModel(500, 300)
model(training_sample[0]).shape

torch.Size([15104, 30000])

In [235]:
training_sample[1].reshape(-1).shape

torch.Size([15104])

In [ ]:
# Training loop for model_training
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    save: bool = False,
    save_epoch: int = 0,
    base_path: str = '',
    save_model_name: str = '',
    save_optimizer_name : str = ''
) -> float:
    """Run one training epoch."""
    if save:
        torch.save(model.state_dict(), base_path + f"Epoch #{save_epoch}" + "-" + save_model_name)
        torch.save(optimizer.state_dict(), base_path + f"Epoch #{save_epoch}" + "-" + save_optimizer_name)
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for inputs, targets, lengths in tqdm(dataloader, desc="Epoch training"):
        inputs, targets, lengths = inputs.to(device), targets.to(device), lengths.to(device = 'cpu')
        optimizer.zero_grad()

        outputs = model(inputs, lengths = lengths)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

### Inference